# Final Project: Mini Drone Parcel Route
AIF1303 · Final Project · Akhmetzhan Turlybek · Group 1 · AI-use level: AI-3

**Scenario.** A delivery drone must carry one parcel across a small 6×6 town map. Before take-off it receives a weather alert and must decide whether to fly now or wait.

Three course algorithms, each feeding the next:

| Step | Algorithm (course week) | Question it answers | Output passed on |
|---|---|---|---|
| 1 | **A\* search** (W2) | What is the shortest route around the obstacles? | route length **L** |
| 2 | **Bayesian inference** (W6) | After the alert, how likely is strong wind? | belief **b = P(Windy \| alert)** |
| 3 | **MDP, one Bellman update** (W5) | Fly now or wait? | the action |

All numbers are invented toy values. Run all cells top to bottom; the last section has sliders.

## 0. Inputs (every assumption in one place)

In [ ]:
# --- Map (row, col); row 0 is the top row ---
ROWS, COLS = 6, 6
START = (0, 0)                        # depot
GOAL = (5, 5)                         # customer
OBSTACLES = {(1, 1), (2, 3), (4, 4)}  # 3 buildings the drone cannot enter
# Moves: up, down, left, right. Each move costs 1 (one unit of battery).

# --- Weather sensor (Bayes) ---
PRIOR_WINDY = 0.30        # P(Windy) before any alert
P_ALERT_IF_WINDY = 0.80   # P(alert | Windy): the sensor catches wind 80% of the time
P_ALERT_IF_CALM = 0.20    # P(alert | Calm): false alarm rate
ALERT_RECEIVED = True     # the drone did receive an alert

# --- Rewards (MDP) ---
DELIVERY_REWARD = 20      # parcel delivered
CRASH_PENALTY = -20       # flying in strong wind: parcel lost
WAIT_COST = -2            # waiting delays the delivery
# R(Calm, Fly) = DELIVERY_REWARD - L   (each step of the A* route costs 1)

## 1. A\* search: shortest route

- **Input:** the grid, start, goal, obstacles.
- **Steps:** always expand the open node with the smallest **f(n) = g(n) + h(n)**, where g = steps taken so far and h = Manhattan distance to the goal, |r − r_goal| + |c − c_goal|.
- **Why A\*:** Manhattan distance never overestimates on a 4-direction grid with step cost 1 (it is *admissible*), so A\* returns a shortest route while expanding fewer nodes than uninformed search. On ties in f, the node closer to the goal (smaller h) goes first.

In [ ]:
import heapq

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def neighbours(cell, obstacles):
    r, c = cell
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:   # up, down, left, right
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and (nr, nc) not in obstacles:
            yield (nr, nc)

def a_star(start, goal, obstacles, use_heuristic=True):
    h = (lambda n: manhattan(n, goal)) if use_heuristic else (lambda n: 0)
    counter = 0                                   # last tie-breaker: first pushed, first out
    frontier = [(h(start), h(start), counter, start)]   # (f, h, counter, node): equal f -> closer to goal first
    g = {start: 0}
    parent = {start: None}
    expanded = []
    while frontier:
        f, _, _, node = heapq.heappop(frontier)
        if node in expanded:
            continue                              # an older, worse copy of this node
        expanded.append(node)
        if node == goal:                          # goal test when expanded -> optimal
            path = []
            while node is not None:
                path.append(node)
                node = parent[node]
            return path[::-1], expanded
        for nb in neighbours(node, obstacles):
            new_g = g[node] + 1
            if nb not in g or new_g < g[nb]:
                g[nb] = new_g
                parent[nb] = node
                counter += 1
                heapq.heappush(frontier, (new_g + h(nb), h(nb), counter, nb))
    return None, expanded                         # no route exists

def draw_ascii(path, obstacles):
    for r in range(ROWS):
        row = []
        for c in range(COLS):
            cell = (r, c)
            row.append("S" if cell == START else "G" if cell == GOAL else
                       "#" if cell in obstacles else "*" if path and cell in path else ".")
        print(" ".join(row))

path, expanded = a_star(START, GOAL, OBSTACLES)
L = len(path) - 1
print("Route:", path)
print("Route length L =", L, "steps")
print("Nodes expanded by A*:", len(expanded))
_, expanded_ucs = a_star(START, GOAL, OBSTACLES, use_heuristic=False)
print("Nodes expanded without the heuristic (h = 0, uniform-cost search):", len(expanded_ucs))
print()
draw_ascii(path, OBSTACLES)
print("\nA* found a shortest route of", L, "steps; L sets the battery cost of flying in step 3.")

Route: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)]
Route length L = 10 steps
Nodes expanded by A*: 11
Nodes expanded without the heuristic (h = 0, uniform-cost search): 33

S . . . . .
* # . . . .
* . . # . .
* . . . . .
* . . . # .
* * * * * G

A* found a shortest route of 10 steps; L sets the battery cost of flying in step 3.


## 2. Bayesian inference: is it windy?

- **Input:** prior P(Windy) = 0.30, sensor P(alert | Windy) = 0.80, P(alert | Calm) = 0.20, and the alert.
- **Steps (Bayes' rule):**

  P(Windy | alert) = P(alert | Windy)·P(Windy) / [P(alert | Windy)·P(Windy) + P(alert | Calm)·P(Calm)]

  = 0.80·0.30 / (0.80·0.30 + 0.20·0.70) = 0.24 / 0.38 = **0.6316**
- Same numbers as the Practical 6 obstacle sensor.

In [ ]:
def bayes(prior, p_obs_if_true, p_obs_if_false):
    # Posterior P(true | observation) after one observation.
    score_true = prior * p_obs_if_true
    score_false = (1 - prior) * p_obs_if_false
    return score_true / (score_true + score_false)

def wind_belief(prior, p_alert_windy, p_alert_calm, alert):
    if alert:
        return bayes(prior, p_alert_windy, p_alert_calm)
    return bayes(prior, 1 - p_alert_windy, 1 - p_alert_calm)   # "no alert" observation

b = wind_belief(PRIOR_WINDY, P_ALERT_IF_WINDY, P_ALERT_IF_CALM, ALERT_RECEIVED)
print(f"score Windy = {PRIOR_WINDY} x {P_ALERT_IF_WINDY} = {PRIOR_WINDY * P_ALERT_IF_WINDY:.2f}")
print(f"score Calm  = {1 - PRIOR_WINDY:.2f} x {P_ALERT_IF_CALM} = {(1 - PRIOR_WINDY) * P_ALERT_IF_CALM:.2f}")
print(f"P(Windy | alert) = b = {b:.4f}   P(Calm | alert) = {1 - b:.4f}")
print(f"\nThe alert raised the wind belief from {PRIOR_WINDY:.2f} to {b:.4f}; b is the MDP's weather belief in step 3.")

score Windy = 0.3 x 0.8 = 0.24
score Calm  = 0.70 x 0.2 = 0.14
P(Windy | alert) = b = 0.6316   P(Calm | alert) = 0.3684

The alert raised the wind belief from 0.30 to 0.6316; b is the MDP's weather belief in step 3.


## 3. MDP: fly now or wait?

- **States:** Calm, Windy. **Actions:** Fly, Wait.
- **Rewards R(s, a):**

  | | Fly | Wait |
  |---|---|---|
  | Calm | 20 − L = 10 (delivered, minus battery for the A\* route) | −2 |
  | Windy | −20 (parcel lost) | −2 |

- **One Bellman update** from V₀ = 0, as the brief allows: Q(s, a) = R(s, a) + γ·Σ P(s′|s, a)·V₀(s′) = R(s, a), because V₀ = 0.
- The drone does not know the state, so it weights each state by the Bayes belief b:

  Q(Fly) = (1 − b)·R(Calm, Fly) + b·R(Windy, Fly) = 0.3684·10 + 0.6316·(−20) = **−8.95**
  Q(Wait) = (1 − b)·(−2) + b·(−2) = **−2**  → pick the larger: **Wait**

In [ ]:
def reward_table(L):
    return {
        ("Calm", "Fly"): DELIVERY_REWARD - L,
        ("Windy", "Fly"): CRASH_PENALTY,
        ("Calm", "Wait"): WAIT_COST,
        ("Windy", "Wait"): WAIT_COST,
    }

def choose_action(b, L):
    R = reward_table(L)
    belief = {"Calm": 1 - b, "Windy": b}
    Q = {a: sum(belief[s] * R[(s, a)] for s in belief) for a in ["Fly", "Wait"]}
    return max(Q, key=Q.get), Q, R

action, Q, R = choose_action(b, L)
for (s, a), r in R.items():
    print(f"R({s}, {a}) = {r}")
print(f"\nQ(Fly)  = {1 - b:.4f} x {R[('Calm', 'Fly')]} + {b:.4f} x {R[('Windy', 'Fly')]} = {Q['Fly']:.2f}")
print(f"Q(Wait) = {1 - b:.4f} x {WAIT_COST} + {b:.4f} x {WAIT_COST} = {Q['Wait']:.2f}")
print("\nDecision:", action.upper())

# Break-even belief: Q(Fly) = Q(Wait)  ->  (1-b)(20-L) - 20b = -2
b_star = (R[("Calm", "Fly")] - WAIT_COST) / (R[("Calm", "Fly")] - CRASH_PENALTY)
print(f"Fly only if P(Windy) < {b_star:.2f}. Here b = {b:.4f}, so the drone waits.")

R(Calm, Fly) = 10
R(Windy, Fly) = -20
R(Calm, Wait) = -2
R(Windy, Wait) = -2

Q(Fly)  = 0.3684 x 10 + 0.6316 x -20 = -8.95
Q(Wait) = 0.3684 x -2 + 0.6316 x -2 = -2.00

Decision: WAIT
Fly only if P(Windy) < 0.40. Here b = 0.6316, so the drone waits.


## 4. Whole pipeline and test cases
`run_mission` chains the three steps. The tests check the hand-calculated values above.

In [ ]:
def run_mission(obstacles=OBSTACLES, prior=PRIOR_WINDY, p_alert_windy=P_ALERT_IF_WINDY,
                p_alert_calm=P_ALERT_IF_CALM, alert=ALERT_RECEIVED):
    path, expanded = a_star(START, GOAL, obstacles)
    if path is None:
        return {"path": None, "L": None, "b": None, "action": "Wait (no route)", "Q": None}
    L = len(path) - 1
    b = wind_belief(prior, p_alert_windy, p_alert_calm, alert)
    action, Q, _ = choose_action(b, L)
    return {"path": path, "L": L, "expanded": len(expanded), "b": b, "action": action, "Q": Q}

# Test 1: the main scenario (alert received)
m = run_mission()
assert m["L"] == 10                       # Manhattan distance 5 + 5, obstacles force no detour
assert round(m["b"], 4) == 0.6316         # 0.24 / 0.38
assert round(m["Q"]["Fly"], 2) == -8.95 and m["Q"]["Wait"] == -2
assert m["action"] == "Wait"

# Test 2: no alert -> belief drops, the drone flies
m2 = run_mission(alert=False)
assert round(m2["b"], 4) == 0.0968        # 0.06 / 0.62
assert m2["action"] == "Fly"

# Test 3: two long walls force a zigzag detour -> longer route, lower Fly value
wall = {(1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5)}
m3 = run_mission(obstacles=wall, alert=False)
assert m3["L"] == 20                      # right, down, all the way left, down, right
assert m3["Q"]["Fly"] < m2["Q"]["Fly"]    # same weather, longer route -> flying is worth less

# Test 4: goal fully blocked -> no route, the drone waits
m4 = run_mission(obstacles={(4, 5), (5, 4)})
assert m4["path"] is None and m4["action"].startswith("Wait")

# Test 5: A* path is never longer than the uninformed search path
for obs in [OBSTACLES, wall]:
    p1, _ = a_star(START, GOAL, obs)
    p2, _ = a_star(START, GOAL, obs, use_heuristic=False)
    assert len(p1) == len(p2)

print("All 5 tests passed.")
for name, r in [("alert", m), ("no alert", m2), ("detour wall, no alert", m3)]:
    print(f"{name:22} L = {r['L']:2}  b = {r['b']:.4f}  Q(Fly) = {r['Q']['Fly']:6.2f}  -> {r['action']}")

All 5 tests passed.
alert                  L = 10  b = 0.6316  Q(Fly) =  -8.95  -> Wait
no alert               L = 10  b = 0.0968  Q(Fly) =   7.10  -> Fly
detour wall, no alert  L = 20  b = 0.0968  Q(Fly) =  -1.94  -> Fly


## 5. Interactive demo
Move a slider and all three algorithms re-run. Obstacles: type cells as `row,col` separated by spaces.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()  # Colab needs this for the sliders to respond
except ImportError:
    pass  # not in Colab; Jupyter shows the sliders without this


In [8]:
import matplotlib.pyplot as plt
import ipywidgets as widgets

def parse_obstacles(text):
    cells = set()
    for part in text.split():
        r, c = part.split(",")
        cell = (int(r), int(c))
        if cell not in (START, GOAL) and 0 <= cell[0] < ROWS and 0 <= cell[1] < COLS:
            cells.add(cell)
    return cells

def demo(obstacles="1,1 2,3 4,4", prior=0.30, p_alert_windy=0.80, p_alert_calm=0.20, alert=True):
    try:
        obs = parse_obstacles(obstacles)
    except ValueError:
        print("Write obstacles like: 1,1 2,3 4,4")
        return
    m = run_mission(obs, prior, p_alert_windy, p_alert_calm, alert)
    fig, ax = plt.subplots(figsize=(4, 4))
    for r in range(ROWS):
        for c in range(COLS):
            colour = "black" if (r, c) in obs else "white"
            ax.add_patch(plt.Rectangle((c, ROWS - 1 - r), 1, 1, facecolor=colour, edgecolor="grey"))
    if m["path"]:
        xs = [c + 0.5 for r, c in m["path"]]
        ys = [ROWS - 1 - r + 0.5 for r, c in m["path"]]
        ax.plot(xs, ys, "-o", color="tab:blue")
    ax.text(START[1] + 0.5, ROWS - 1 - START[0] + 0.5, "S", ha="center", va="center", fontsize=14, color="green", bbox=dict(facecolor="white", edgecolor="none"))
    ax.text(GOAL[1] + 0.5, ROWS - 1 - GOAL[0] + 0.5, "G", ha="center", va="center", fontsize=14, color="red", bbox=dict(facecolor="white", edgecolor="none"))
    ax.set_xlim(0, COLS); ax.set_ylim(0, ROWS); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("A* route" if m["path"] else "No route")
    plt.show()
    if m["path"] is None:
        print("1. A*: no route to the customer.  ->  Decision: WAIT")
        return
    print(f"1. A*:    route length L = {m['L']}  (nodes expanded: {m['expanded']})")
    print(f"2. Bayes: P(Windy | {'alert' if alert else 'no alert'}) = {m['b']:.4f}")
    print(f"3. MDP:   Q(Fly) = {m['Q']['Fly']:.2f}   Q(Wait) = {m['Q']['Wait']:.2f}   ->  Decision: {m['action'].upper()}")

widgets.interact(
    demo,
    obstacles=widgets.Text(value="1,1 2,3 4,4", description="Obstacles"),
    prior=widgets.FloatSlider(value=0.30, min=0.01, max=0.99, step=0.01, description="P(Windy)"),
    p_alert_windy=widgets.FloatSlider(value=0.80, min=0.01, max=0.99, step=0.01, description="P(alert|W)"),
    p_alert_calm=widgets.FloatSlider(value=0.20, min=0.01, max=0.99, step=0.01, description="P(alert|C)"),
    alert=widgets.Checkbox(value=True, description="Alert received"),
);

interactive(children=(Text(value='1,1 2,3 4,4', description='Obstacles'), FloatSlider(value=0.3, description='…

## 6. Results, evaluation and possible improvements

**Results.** A\* finds a 10-step route and expands 11 nodes; the same search without the heuristic expands 33. The alert raises P(Windy) from 0.30 to 0.6316. With that belief, Q(Fly) = −8.95 < Q(Wait) = −2, so the drone **waits**. Without the alert, P(Windy) = 0.0968 and the drone **flies** (Q(Fly) = 7.10).

**How the parts work together.** A\* gives the route length, which sets the flying reward. Bayes gives the wind belief, which weights the rewards. The MDP combines both into one decision. Change any one input and the decision can change: the break-even belief is P(Windy) = 0.40 for a 10-step route.

**Limits.** All probabilities and rewards are invented. One sensor reading; one decision step; wind assumed equal over the whole map.

**Possible improvements.**
1. Run full value iteration with weather transitions (Calm ↔ Windy over time), so waiting has a real future value.
2. Use an HMM to track the wind over several alerts instead of one.
3. Give each cell its own wind risk and let A\* avoid windy cells (cost = steps + risk).